# Stage 1 - Unsupervised Anomaly Baseline

Pipeline: pre-processing -> SLIC/GLCM feature extraction -> normal-water baseline -> deviation scoring -> scored evaluation on negative controls and positive datasets (prawn / tuna / cetacean) -> threshold and component-size tuning -> false-positive diagnostics -> final scored summary.

In [2]:
# Cell 1: Imports and config
import cv2
import numpy as np
import random
import json
from pathlib import Path
from skimage.segmentation import slic
from skimage.feature import graycomatrix, graycoprops
from scipy import ndimage
from concurrent.futures import ProcessPoolExecutor

# Dataset paths
BASELINE_DIR  = Path("data/normal_sea/baseline_split")   # builds "what is normal water"
NEGATIVE_DIR  = Path("data/normal_sea/heldout_split")     # held-out - never touched during baseline build
PRAWN_DIR     = Path("data/prawns")             # deduplicated 42 unique prawn images, not the raw 82
TUNA_DIR      = Path("data/tuna")                         # optional bonus positive set
CETACEAN_DIR  = Path("data/cetacean")                     # optional bonus positive set

IMG_SIZE = (512, 512)
N_SEGMENTS = 150
GLCM_LEVELS = 32
BASELINE_SAMPLE_SIZE = 300     # confirmed via Cell 4a convergence check
MIN_COMPONENT_SIZE = 3         # minimum flagged superpixels to count as a real detection
Z_THRESH = 2.5

random.seed(42)

In [3]:
# Cell 2: Pre-processing functions 
# Dataset used: none - these are applied identically to every set later.

def load_and_resize(path, size=IMG_SIZE):
    img = cv2.imread(str(path))
    img = cv2.resize(img, size)
    return img

def suppress_glint(img_bgr, thresh=220, dilate_iter=2):
    """Masks out bright sun-glint pixels on the L channel."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L = lab[:, :, 0]
    glint_mask = (L > thresh).astype(np.uint8)
    glint_mask = cv2.dilate(glint_mask, np.ones((5, 5), np.uint8), iterations=dilate_iter)
    return glint_mask  # 1 = glint, exclude from analysis

def to_lab(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

def preprocess_image(path):
    """Full chain: load -> resize -> LAB -> glint mask."""
    img = load_and_resize(path)
    lab = to_lab(img)
    glint_mask = suppress_glint(img)
    return img, lab, glint_mask

In [4]:
# Cell 3: Fast SLIC + GLCM feature extraction 
# Dataset used: none - reusable functions for every image below.

def quantize(patch, levels=GLCM_LEVELS):
    return (patch.astype(np.float32) / 256 * levels).astype(np.uint8)

def extract_superpixel_features(img_lab, glint_mask, n_segments=N_SEGMENTS):
    """
    Runs SLIC on the L channel, then computes GLCM texture stats per
    superpixel, keeping true 2D spatial structure (fix vs. the inherited
    flattened-1D bug). Uses ndimage.find_objects for speed instead of a
    per-superpixel full-image boolean comparison, and quantizes grey
    levels down from 256 to reduce GLCM matrix size.
    """
    L_channel = img_lab[:, :, 0]
    segments = slic(img_lab, n_segments=n_segments, compactness=10, start_label=1)
    objects = ndimage.find_objects(segments)

    features = []
    for seg_id, bbox in enumerate(objects, start=1):
        if bbox is None:
            continue
        y_slice, x_slice = bbox
        local_mask = segments[y_slice, x_slice] == seg_id

        if glint_mask[y_slice, x_slice][local_mask].mean() > 0.5:
            continue  # skip glint-dominated superpixels entirely

        patch = L_channel[y_slice, x_slice]
        if patch.shape[0] < 2 or patch.shape[1] < 2:
            continue

        patch_q = quantize(patch)
        glcm = graycomatrix(patch_q, distances=[1], angles=[0],
                             levels=GLCM_LEVELS, symmetric=True, normed=True)

        features.append({
            "seg_id": int(seg_id),
            "bbox": (x_slice.start, y_slice.start, x_slice.stop, y_slice.stop),
            "contrast": float(graycoprops(glcm, 'contrast')[0, 0]),
            "homogeneity": float(graycoprops(glcm, 'homogeneity')[0, 0]),
            "energy": float(graycoprops(glcm, 'energy')[0, 0]),
        })
    return segments, features

def process_one_image(path):
    """Wrapper for parallel/serial use - returns just the feature list."""
    img, lab, glint_mask = preprocess_image(path)
    _, feats = extract_superpixel_features(lab, glint_mask)
    return feats

In [5]:
# Cell 4a: Convergence check
# Dataset used: normal sea - baseline split, sampled at increasing sizes.
# Confirms BASELINE_SAMPLE_SIZE is large enough before committing to it.

all_baseline_paths = list(BASELINE_DIR.glob("*.jpg"))
print(f"Total normal sea baseline images available: {len(all_baseline_paths)}")

for n in [100, 300, 600]:
    sample = random.sample(all_baseline_paths, min(n, len(all_baseline_paths)))
    stats = {"contrast": [], "homogeneity": [], "energy": []}
    for path in sample:
        feats = process_one_image(path)
        for f in feats:
            for k in stats:
                stats[k].append(f[k])
    means = {k: round(float(np.mean(v)), 4) for k, v in stats.items()}
    print(f"n={n}: {means}")

Total normal sea baseline images available: 3283
n=100: {'contrast': 0.3248, 'homogeneity': 0.8821, 'energy': 0.4117}
n=300: {'contrast': 0.347, 'homogeneity': 0.8763, 'energy': 0.4136}
n=600: {'contrast': 0.3434, 'homogeneity': 0.8793, 'energy': 0.4135}


In [6]:
# Cell 4b: Build the normal-water baseline
# Dataset used: normal sea - baseline split, subsampled to BASELINE_SAMPLE_SIZE.
# This must never see negative controls or positives.

baseline_paths = random.sample(all_baseline_paths, min(BASELINE_SAMPLE_SIZE, len(all_baseline_paths)))

baseline_stats = {"contrast": [], "homogeneity": [], "energy": []}
for path in baseline_paths:
    feats = process_one_image(path)
    for f in feats:
        for k in baseline_stats:
            baseline_stats[k].append(f[k])

baseline_ref = {k: (np.mean(v), np.std(v)) for k, v in baseline_stats.items()}
print(f"Baseline built from {len(baseline_paths)} images")
print("Baseline (mean, std):", baseline_ref)

# If this is too slow serially on your machine, swap the loop above for:
# with ProcessPoolExecutor() as executor:
#     for feats in executor.map(process_one_image, baseline_paths):
#         for f in feats:
#             for k in baseline_stats:
#                 baseline_stats[k].append(f[k])

Baseline built from 300 images
Baseline (mean, std): {'contrast': (np.float64(0.3537315776465872), np.float64(0.7140890982405314)), 'homogeneity': (np.float64(0.8764585502779129), np.float64(0.07960758882246824)), 'energy': (np.float64(0.4092685737816868), np.float64(0.16064229714671144))}


In [7]:
# Cell 5: Deviation scoring + detector runner
# Dataset used: none - reusable scorer for the runs below.

def score_region_deviation(feat, baseline_ref, z_thresh=Z_THRESH):
    z_scores = {
        k: abs(feat[k] - baseline_ref[k][0]) / (baseline_ref[k][1] + 1e-6)
        for k in ["contrast", "homogeneity", "energy"]
    }
    max_z = max(z_scores.values())
    return max_z > z_thresh, max_z

def run_detector(img_dir, baseline_ref, sample_size=None,
                  extensions=("*.jpg", "*.jpeg", "*.png"),
                  min_component_size=MIN_COMPONENT_SIZE,
                  z_thresh=Z_THRESH):
    paths = []
    for ext in extensions:
        paths.extend(Path(img_dir).glob(ext))

    if sample_size:
        paths = random.sample(paths, min(sample_size, len(paths)))

    if len(paths) == 0:
        raise ValueError(f"No images found in {img_dir} - check path and file extensions")

    results = {}
    for path in paths:
        img, lab, glint_mask = preprocess_image(path)
        segments, feats = extract_superpixel_features(lab, glint_mask)
        flagged = [f for f in feats if score_region_deviation(f, baseline_ref, z_thresh)[0]]
        # Only counts as a detection if enough flagged superpixels form a real region,
        # not just a single stray one.
        is_detected = len(flagged) >= min_component_size
        results[path.name] = {
            "n_flagged": len(flagged),
            "is_detected": is_detected,
            "flagged_regions": flagged,
        }
    return results

def safe_detection_rate(results, label=""):
    if len(results) == 0:
        raise ValueError(f"No results to compute detection rate for: {label}")
    return np.mean([1 if r["is_detected"] else 0 for r in results.values()])

In [8]:
# Cell 6: Run on negative controls
# Dataset used: normal sea - heldout split. Checks the detector doesn't
# cry wolf on plume-free water it has never seen.

negative_results = run_detector(NEGATIVE_DIR, baseline_ref, sample_size=500)
false_positive_rate = safe_detection_rate(negative_results, "negative controls")
print(f"False-positive rate on plume-free water (n={len(negative_results)}): {false_positive_rate:.2%}")

False-positive rate on plume-free water (n=457): 25.82%


In [9]:
# Cell 7: Run on prawn positives
# Dataset used: prawn dataset - 42 deduplicated unique images.
# Main positive answer-key set for Stage 1.

positive_results = run_detector(PRAWN_DIR, baseline_ref)
prawn_detection_rate = safe_detection_rate(positive_results, "prawn positives")
print(f"Detection rate on prawn plumes (n={len(positive_results)}): {prawn_detection_rate:.2%}")

Detection rate on prawn plumes (n=82): 91.46%


In [10]:
# Cell 8:tuna and cetacean as extra positive checks 
# Dataset used: tuna and cetacean datasets. Scored separately per target
# type since these are more discrete/higher-contrast targets than the
# diffuse prawn plumes, so detection rates aren't expected to match exactly.
# Comment this cell out if TUNA_DIR / CETACEAN_DIR aren't populated yet,
# and remove the matching keys from the Cell 10 summary.

tuna_results = run_detector(TUNA_DIR, baseline_ref)
cetacean_results = run_detector(CETACEAN_DIR, baseline_ref)

tuna_detection_rate = safe_detection_rate(tuna_results, "tuna")
cetacean_detection_rate = safe_detection_rate(cetacean_results, "cetacean")

print(f"Detection rate on tuna: {tuna_detection_rate:.2%}")
print(f"Detection rate on cetacean: {cetacean_detection_rate:.2%}")

Detection rate on tuna: 97.58%
Detection rate on cetacean: 98.36%


In [11]:
# Cell 9: Cache raw features for negatives and prawn positives
# Dataset used: normal sea (heldout) and prawn (unique 42).
# Extracts features once so the threshold and component-size sweeps below
# can re-score cheaply without re-running SLIC/GLCM every time.

def run_and_cache_features(img_dir, extensions=("*.jpg", "*.jpeg", "*.png"), sample_size=None):
    paths = []
    for ext in extensions:
        paths.extend(Path(img_dir).glob(ext))
    if sample_size:
        paths = random.sample(paths, min(sample_size, len(paths)))
    cache = {}
    for path in paths:
        img, lab, glint_mask = preprocess_image(path)
        _, feats = extract_superpixel_features(lab, glint_mask)
        cache[path.name] = feats
    return cache

neg_cache = run_and_cache_features(NEGATIVE_DIR, sample_size=500)
pos_cache = run_and_cache_features(PRAWN_DIR)
print(f"Cached features for {len(neg_cache)} negative images and {len(pos_cache)} prawn images")

Cached features for 457 negative images and 82 prawn images


In [12]:
# Cell 10: Threshold sweep (z_thresh)
# Dataset used: cached negative + prawn features from Cell 9.
# Shows the false-positive vs. detection trade-off across thresholds,
# instead of guessing a single value.

def rate_at(cache, z, min_size):
    return np.mean([
        1 if sum(1 for f in feats if score_region_deviation(f, baseline_ref, z)[0]) >= min_size else 0
        for feats in cache.values()
    ])

print(f"{'z_thresh':>10} | {'false_positive_rate':>20} | {'prawn_detection_rate':>20}")
for z in [1.5, 2.0, 2.5, 3.0, 3.5]:
    neg_rate = rate_at(neg_cache, z, MIN_COMPONENT_SIZE)
    pos_rate = rate_at(pos_cache, z, MIN_COMPONENT_SIZE)
    print(f"{z:>10} | {neg_rate:>19.2%} | {pos_rate:>19.2%}")

  z_thresh |  false_positive_rate | prawn_detection_rate
       1.5 |              50.11% |             100.00%
       2.0 |              36.54% |              96.34%
       2.5 |              25.82% |              91.46%
       3.0 |              18.38% |              84.15%
       3.5 |              14.66% |              75.61%


In [13]:
# Cell 11: Component-size sweep (independent of z_thresh)
# Dataset used: cached negative + prawn features from Cell 9.
# Sweeps min_component_size at the current Z_THRESH. Try this alongside
# Cell 10 rather than tuning one knob at a time, since they interact.

print(f"{'min_component_size':>20} | {'false_positive_rate':>20} | {'prawn_detection_rate':>20}")
for min_size in [3, 5, 8, 10]:
    neg_rate = rate_at(neg_cache, Z_THRESH, min_size)
    pos_rate = rate_at(pos_cache, Z_THRESH, min_size)
    print(f"{min_size:>20} | {neg_rate:>19.2%} | {pos_rate:>19.2%}")

  min_component_size |  false_positive_rate | prawn_detection_rate
                   3 |              25.82% |              91.46%
                   5 |              21.23% |              85.37%
                   8 |              18.82% |              78.05%
                  10 |              17.94% |              65.85%


In [14]:
# Cell 12: Inspect false positives directly
# Dataset used: normal sea - heldout split (the false-positive images
# themselves from Cell 6's negative_results).
# Prints where the flagged regions land, so you can check whether they
# cluster near edges/horizon/bright patches (a sign glint suppression or
# ROI masking isn't catching everything) rather than being a fundamental
# flaw in the texture-deviation logic.

print("Sample false positives (image, n_flagged, bounding boxes of flagged regions):")
shown = 0
for name, r in negative_results.items():
    if r["is_detected"]:
        print(name, r["n_flagged"], [f["bbox"] for f in r["flagged_regions"]])
        shown += 1
    if shown >= 5:
        break

if shown == 0:
    print("No false positives found in negative_results - re-run Cell 6 first.")

Sample false positives (image, n_flagged, bounding boxes of flagged regions):
noisy_102_png.rf.bb199ef1990fcead4d3e2aabde559b20.jpg 53 [(0, 0, 161, 125), (143, 0, 200, 47), (331, 0, 375, 44), (417, 0, 465, 45), (139, 38, 198, 93), (336, 39, 383, 85), (469, 41, 512, 85), (54, 44, 146, 128), (144, 78, 206, 130), (199, 79, 250, 125), (420, 81, 466, 127), (466, 83, 512, 127), (84, 98, 154, 159), (0, 107, 99, 173), (142, 120, 200, 171), (289, 123, 334, 169), (333, 125, 377, 169), (467, 125, 512, 169), (421, 126, 467, 169), (52, 141, 117, 194), (98, 145, 167, 204), (0, 152, 60, 214), (157, 164, 208, 212), (292, 166, 337, 210), (381, 166, 425, 211), (468, 167, 512, 210), (81, 183, 128, 253), (32, 191, 94, 271), (0, 209, 57, 281), (377, 209, 422, 254), (466, 209, 512, 252), (292, 210, 336, 252), (249, 212, 296, 256), (116, 244, 173, 293), (68, 247, 124, 315), (381, 250, 425, 296), (294, 251, 338, 294), (467, 252, 512, 295), (249, 254, 296, 296), (30, 257, 108, 327), (0, 273, 46, 332), (293, 29

In [15]:
# Cell 12b: Visualize a false positive (optional, needs matplotlib)
# Dataset used: same false-positive image(s) as Cell 12, drawn with boxes
# around the flagged regions so you can eyeball whether they land on
# genuine texture, glint, horizon, or image edges.

import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_flagged(img_dir, filename, flagged_regions):
    img_path = Path(img_dir) / filename
    img = load_and_resize(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(img_rgb)
    for f in flagged_regions:
        x0, y0, x1, y1 = f["bbox"]
        rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                  linewidth=1.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
    ax.set_title(f"Flagged regions: {filename}")
    ax.axis('off')
    plt.show()

# Visualize the first false positive found
for name, r in negative_results.items():
    if r["is_detected"]:
        visualize_flagged(NEGATIVE_DIR, name, r["flagged_regions"])
        break


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\bcura\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\bcura\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\bcura\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\bcura\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: initialization failed

In [18]:
# Cell 13: Combine into the scored summary (the deliverable)
# Dataset used: all of them - pulls every run together into the final
# scored result that counts as "Stage 1 complete".
# Update z_thresh_used / min_component_size_used if you changed Z_THRESH
# or MIN_COMPONENT_SIZE in Cell 1 based on the sweeps above, and re-run
# Cells 6-8 with the final values before generating this summary.

summary = {
    "baseline_images_used": len(baseline_paths),
    "z_thresh_used": Z_THRESH,
    "min_component_size_used": MIN_COMPONENT_SIZE,
    "false_positive_rate": false_positive_rate,
    "detection_rate_by_target": {
        "prawn": prawn_detection_rate,
        "tuna": tuna_detection_rate,
        "cetacean": cetacean_detection_rate,
    },
    "n_negative_images": len(negative_results),
    "n_prawn_images": len(positive_results),
    "n_tuna_images": len(tuna_results),
    "n_cetacean_images": len(cetacean_results),
}
print(json.dumps(summary, indent=2))

with open("stage1_scored_baseline.json", "w") as f:
    json.dump(summary, f, indent=2)

{
  "baseline_images_used": 300,
  "z_thresh_used": 2.5,
  "min_component_size_used": 3,
  "false_positive_rate": 0.25820568927789933,
  "detection_rate_by_target": {
    "prawn": 0.9146341463414634,
    "tuna": 0.9757698132256436,
    "cetacean": 0.9836461126005361
  },
  "n_negative_images": 457,
  "n_prawn_images": 82,
  "n_tuna_images": 1981,
  "n_cetacean_images": 7460
}
